# Stimulus Validation Materials

This notebook generates the blinded human-validation materials for the original 20-pair behavioral pilot.

It creates two independent validation instruments:

1. **Sentence-level ratings** for formality, assertiveness/confidence, politeness, naturalness, perceived expertise/authority, and evidential strength.
2. **Pair-level semantic-equivalence ratings** for plain/formal versions.

The pilot stimuli themselves are not modified here. This notebook only prepares validation materials.

## Design principles

- 20 matched plain/formal pairs = 40 sentence-level stimuli.
- Sentence order is randomized with a fixed seed.
- Matched plain/formal versions are not adjacent.
- Raters do not see truth status, register labels, domains, or hypotheses.
- Semantic-equivalence items are separately randomized.
- Left/right presentation of plain/formal versions is randomized.
- Stable neutral IDs (`S001`–`S040`, `E001`–`E020`) are used for later response parsing.

## Provenance note

This notebook was reconstructed from the documented Phase 3 validation workflow after the original notebook file was unavailable.

The reconstruction preserves the analysis design and fixed randomization seeds used for the project.


In [ ]:
import os
import glob
import numpy as np
import pandas as pd

SENTENCE_SEED = 42
SEMANTIC_SEED = 84

print("Libraries loaded.")

## Load the frozen pilot stimulus bank

The expected input file is `pilot_20.csv`.

The notebook searches several common locations so that it can work in Colab or in a local repository checkout.


In [ ]:
candidate_paths = [
    "/content/pilot_20.csv",
    "pilot_20.csv",
    "../data/pilot_20.csv",
    "data/pilot_20.csv"
]

PILOT_FILE = None

for path in candidate_paths:
    if os.path.exists(path):
        PILOT_FILE = path
        break

if PILOT_FILE is None:
    matches = glob.glob(
        "/content/**/pilot_20.csv",
        recursive=True
    )

    if matches:
        PILOT_FILE = matches[0]

if PILOT_FILE is None:
    raise FileNotFoundError(
        "pilot_20.csv was not found. "
        "Upload it to Colab or place it in the repository data folder."
    )

pilot = pd.read_csv(
    PILOT_FILE,
    dtype={"item_id": str}
)

pilot["item_id"] = (
    pilot["item_id"]
    .str.zfill(3)
)

required = [
    "item_id",
    "domain",
    "truth",
    "plain",
    "formal"
]

missing = [
    column
    for column in required
    if column not in pilot.columns
]

if missing:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(missing)
    )

print("Loaded:", PILOT_FILE)
print("Rows:", len(pilot))
print(
    "Unique item IDs:",
    pilot["item_id"].nunique()
)

display(
    pilot.head()
)

## Sentence-level validation material

Each of the 20 pilot pairs contributes two sentences:

- plain
- formal

The 40 sentences are randomized with seed `42`.

Truth status, register, domain, and pair identity are retained only in the hidden master.


In [ ]:
sentence_rows = []

for _, row in pilot.iterrows():

    sentence_rows.append({
        "pair_id": row["item_id"],
        "domain": row["domain"],
        "truth": row["truth"],
        "register": "plain",
        "sentence": row["plain"]
    })

    sentence_rows.append({
        "pair_id": row["item_id"],
        "domain": row["domain"],
        "truth": row["truth"],
        "register": "formal",
        "sentence": row["formal"]
    })

sentence_pool = pd.DataFrame(
    sentence_rows
)

sentence_master = (
    sentence_pool
    .sample(
        frac=1,
        random_state=SENTENCE_SEED
    )
    .reset_index(drop=True)
)

sentence_master.insert(
    0,
    "presentation_order",
    range(
        1,
        len(sentence_master) + 1
    )
)

sentence_master.insert(
    1,
    "stimulus_id",
    [
        f"S{i:03d}"
        for i in range(
            1,
            len(sentence_master) + 1
        )
    ]
)

print(
    "Sentence-level stimuli:",
    len(sentence_master)
)

print(
    "Unique pairs:",
    sentence_master["pair_id"].nunique()
)

display(
    sentence_master.head(10)
)

In [ ]:
adjacent_match = (
    sentence_master["pair_id"]
    == sentence_master["pair_id"].shift(1)
)

print(
    "Adjacent matched-pair problems:",
    int(adjacent_match.sum())
)

if adjacent_match.any():

    raise ValueError(
        "A matched plain/formal pair is adjacent. "
        "Revise the randomization before using the materials."
    )

else:

    print(
        "PASS: No matched plain/formal pair is adjacent."
    )

### Blinded sentence-level rater sheet

Participants see only:

- presentation order;
- neutral stimulus ID;
- sentence;
- six blank 1–7 rating fields.

They do **not** see:

- pair ID;
- domain;
- truth status;
- register condition.


In [ ]:
sentence_rater_sheet = sentence_master[
    [
        "presentation_order",
        "stimulus_id",
        "sentence"
    ]
].copy()

sentence_rater_sheet[
    "formality_1_7"
] = ""

sentence_rater_sheet[
    "assertiveness_confidence_1_7"
] = ""

sentence_rater_sheet[
    "politeness_1_7"
] = ""

sentence_rater_sheet[
    "naturalness_1_7"
] = ""

sentence_rater_sheet[
    "expertise_authority_1_7"
] = ""

sentence_rater_sheet[
    "evidential_strength_1_7"
] = ""

display(
    sentence_rater_sheet.head()
)

## Semantic-equivalence validation material

The 20 plain/formal pairs are randomized independently with seed `84`.

For every pair, the side on which the plain and formal versions appear is randomized.

Participants see only:

- neutral semantic ID;
- Sentence A;
- Sentence B.

They do not see register or truth labels.


In [ ]:
semantic_pairs = (
    pilot
    .sample(
        frac=1,
        random_state=SEMANTIC_SEED
    )
    .reset_index(drop=True)
    .copy()
)

rng = np.random.default_rng(
    SEMANTIC_SEED
)

plain_on_left = rng.choice(
    [True, False],
    size=len(semantic_pairs)
)

semantic_rows = []

for i, row in semantic_pairs.iterrows():

    if plain_on_left[i]:

        sentence_A = row["plain"]
        register_A = "plain"

        sentence_B = row["formal"]
        register_B = "formal"

    else:

        sentence_A = row["formal"]
        register_A = "formal"

        sentence_B = row["plain"]
        register_B = "plain"

    semantic_rows.append({
        "presentation_order": i + 1,
        "semantic_id": f"E{i + 1:03d}",
        "pair_id": row["item_id"],
        "domain": row["domain"],
        "truth": row["truth"],
        "sentence_A": sentence_A,
        "register_A": register_A,
        "sentence_B": sentence_B,
        "register_B": register_B
    })

semantic_master = pd.DataFrame(
    semantic_rows
)

print(
    "Semantic pairs:",
    len(semantic_master)
)

print(
    "Unique pair IDs:",
    semantic_master["pair_id"].nunique()
)

display(
    semantic_master.head()
)

In [ ]:
semantic_rater_sheet = semantic_master[
    [
        "presentation_order",
        "semantic_id",
        "sentence_A",
        "sentence_B"
    ]
].copy()

semantic_rater_sheet[
    "semantic_equivalence_1_7"
] = ""

semantic_rater_sheet[
    "same_factual_proposition"
] = ""

display(
    semantic_rater_sheet.head()
)

## Validation dimensions

### Sentence-level ratings

Each sentence is rated from 1–7 on:

1. Formality
2. Assertiveness / confidence
3. Politeness
4. Naturalness
5. Perceived expertise / authority
6. Evidential strength

### Semantic-equivalence ratings

Each pair receives:

- semantic equivalence: 1–7;
- same factual proposition: Yes / No / Unsure.

The sentence-level and semantic-equivalence instruments are intended for separate participant groups.


## Save the four validation CSV files


In [ ]:
OUTPUT_DIR = "/content/validation"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

sentence_master_path = os.path.join(
    OUTPUT_DIR,
    "validation_sentence_level_master.csv"
)

sentence_rater_path = os.path.join(
    OUTPUT_DIR,
    "validation_sentence_level_rater_sheet.csv"
)

semantic_master_path = os.path.join(
    OUTPUT_DIR,
    "validation_semantic_master.csv"
)

semantic_rater_path = os.path.join(
    OUTPUT_DIR,
    "validation_semantic_rater_sheet.csv"
)

sentence_master.to_csv(
    sentence_master_path,
    index=False
)

sentence_rater_sheet.to_csv(
    sentence_rater_path,
    index=False
)

semantic_master.to_csv(
    semantic_master_path,
    index=False
)

semantic_rater_sheet.to_csv(
    semantic_rater_path,
    index=False
)

print("Saved:")
print(sentence_master_path)
print(sentence_rater_path)
print(semantic_master_path)
print(semantic_rater_path)

## Final structural checks


In [ ]:
checks = {
    "sentence_master_rows":
        len(sentence_master),

    "sentence_unique_stimulus_ids":
        sentence_master[
            "stimulus_id"
        ].nunique(),

    "sentence_unique_pairs":
        sentence_master[
            "pair_id"
        ].nunique(),

    "sentence_adjacent_pair_problems":
        int(
            (
                sentence_master["pair_id"]
                == sentence_master[
                    "pair_id"
                ].shift(1)
            ).sum()
        ),

    "semantic_master_rows":
        len(semantic_master),

    "semantic_unique_ids":
        semantic_master[
            "semantic_id"
        ].nunique(),

    "semantic_unique_pairs":
        semantic_master[
            "pair_id"
        ].nunique()
}

for key, value in checks.items():
    print(
        f"{key}: {value}"
    )

assert (
    checks[
        "sentence_master_rows"
    ]
    == 40
)

assert (
    checks[
        "sentence_unique_stimulus_ids"
    ]
    == 40
)

assert (
    checks[
        "sentence_unique_pairs"
    ]
    == 20
)

assert (
    checks[
        "sentence_adjacent_pair_problems"
    ]
    == 0
)

assert (
    checks[
        "semantic_master_rows"
    ]
    == 20
)

assert (
    checks[
        "semantic_unique_ids"
    ]
    == 20
)

assert (
    checks[
        "semantic_unique_pairs"
    ]
    == 20
)

print(
    "\nSUCCESS: Validation materials are structurally complete."
)